**Objetivo**

Promover as metas de alfabetização por unidade federativa da camada Bronze para a Silver.

**Fonte de dados**

- `bronze.meta_alfabetizacao_uf`

**Destino**

- `silver.meta_alfabetizacao_uf`

**Granularidade**

- Uma linha por `ano`, `sigla_uf` e `rede`.

> As validações de qualidade são informativas, como no notebook de município. Apenas a ausência de colunas obrigatórias impede tecnicamente a execução.

## 0. Configurando sessão Spark

In [6]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("bronze_to_silver_meta_alfabetizacao_uf")
    .config(
        "spark.jars.packages",
        "com.google.cloud.spark:spark-bigquery-with-dependencies_2.13:0.44.2"
    )
    .getOrCreate()
)

spark.conf.set("parentProject", "tech-challenge-fase-2-505123")

:: loading settings :: url = jar:file:/opt/micromamba/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/jupyter/.ivy2.5.2/cache
The jars for the packages stored in: /home/jupyter/.ivy2.5.2/jars
com.google.cloud.spark#spark-bigquery-with-dependencies_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-f93925b4-70cc-4407-9eb2-84137768d552;1.0
	confs: [default]
	found com.google.cloud.spark#spark-bigquery-with-dependencies_2.13;0.44.2 in central
:: resolution report :: resolve 200ms :: artifacts dl 5ms
	:: modules in use:
	com.google.cloud.spark#spark-bigquery-with-dependencies_2.13;0.44.2 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	-----------------------------------------

In [7]:
spark.conf.set("spark.sql.repl.eagerEval.enabled", True)
spark.conf.set("spark.sql.repl.eagerEval.maxNumRows", 20)
spark.conf.set("spark.sql.repl.eagerEval.truncate", 100)

## 1. Imports

In [8]:
from pyspark.sql import functions as F

## 2. Geração de parâmetros

In [9]:
par_source_project = "tech-challenge-fase-2-505123"
par_source_bronze_meta = f"{par_source_project}.bronze.meta_alfabetizacao_uf"
par_source_silver_meta = f"{par_source_project}.silver.meta_alfabetizacao_uf"

colunas_meta = [f"meta_alfabetizacao_{ano}" for ano in range(2024, 2031)]
colunas_percentuais = ["taxa_alfabetizacao", *colunas_meta, "percentual_participacao"]
colunas_esperadas = [
    "ano", "sigla_uf", "rede", *colunas_percentuais,
    "_ingestao_timestamp", "_fonte"
]

ufs_validas = [
    "AC", "AL", "AP", "AM", "BA", "CE", "DF", "ES", "GO",
    "MA", "MT", "MS", "MG", "PA", "PB", "PR", "PE", "PI",
    "RJ", "RN", "RS", "RO", "RR", "SC", "SP", "SE", "TO",
]
redes_conhecidas = ["Federal", "Estadual", "Municipal", "Privada", "Pública"]

## 3. Leitura dos dados da origem

In [10]:
df_src_meta = (
    spark.read.format("bigquery")
    .option("table", par_source_bronze_meta)
    .load()
)

### 3.1. Validação do contrato de entrada

In [11]:
colunas_ausentes = sorted(set(colunas_esperadas) - set(df_src_meta.columns))
if colunas_ausentes:
    raise ValueError(f"Schema inválido. Colunas ausentes na Bronze: {colunas_ausentes}")

df_meta = df_src_meta.select(*colunas_esperadas)

### 3.2. Diagnóstico dos domínios

In [12]:
for coluna in ["ano", "sigla_uf", "rede"]:
    print(f"=== Domínio de {coluna} ===")
    (
        df_meta.groupBy(coluna).count()
        .orderBy(F.desc("count"), F.asc_nulls_first(coluna))
        .show(100, truncate=False)
    )

=== Domínio de ano ===


+----+-----+
|ano |count|
+----+-----+
|2023|27   |
|2024|27   |
|2025|27   |
+----+-----+

=== Domínio de sigla_uf ===
+--------+-----+
|sigla_uf|count|
+--------+-----+
|AC      |3    |
|AL      |3    |
|AM      |3    |
|AP      |3    |
|BA      |3    |
|CE      |3    |
|DF      |3    |
|ES      |3    |
|GO      |3    |
|MA      |3    |
|MG      |3    |
|MS      |3    |
|MT      |3    |
|PA      |3    |
|PB      |3    |
|PE      |3    |
|PI      |3    |
|PR      |3    |
|RJ      |3    |
|RN      |3    |
|RO      |3    |
|RR      |3    |
|RS      |3    |
|SC      |3    |
|SE      |3    |
|SP      |3    |
|TO      |3    |
+--------+-----+

=== Domínio de rede ===
+-------+-----+
|rede   |count|
+-------+-----+
|Pública|81   |
+-------+-----+



## 4. Transformações

In [13]:
rede_limpa = F.trim(F.col("rede").cast("string"))
rede_minuscula = F.lower(rede_limpa)

df_silver_meta = (
    df_meta
    .withColumn("ano", F.col("ano").cast("int"))
    .withColumn("sigla_uf", F.upper(F.trim(F.col("sigla_uf"))))
    .withColumn(
        "rede",
        F.when(rede_minuscula.isin("pública", "publica", "p�blica", "pãºblica"), F.lit("Pública"))
        .when(rede_limpa == "", F.lit(None))
        .otherwise(F.initcap(rede_minuscula))
    )
    .withColumn("_ingestao_timestamp", F.col("_ingestao_timestamp").cast("timestamp"))
    .withColumn("_fonte", F.trim(F.col("_fonte")))
)

for coluna in colunas_percentuais:
    df_silver_meta = df_silver_meta.withColumn(coluna, F.col(coluna).cast("double"))

### 4.1. Data de carregamento e exclusão de duplicadas

In [14]:
chave = ["ano", "sigla_uf", "rede"]
df_silver_meta_antes_dedup = df_silver_meta
df_silver_meta = (
    df_silver_meta
    .dropDuplicates(chave)
    .withColumn("_silver_timestamp", F.current_timestamp())
)

## 5. Validação da qualidade

In [15]:
print("=== Relatório de Qualidade — silver.meta_alfabetizacao_uf ===")

qtd_bronze = df_src_meta.count()
qtd_silver = df_silver_meta.count()

# 1) Duplicidade na chave natural
dups_antes = (
    df_silver_meta_antes_dedup.groupBy(*chave).count()
    .filter(F.col("count") > 1).count()
)
dups_depois = (
    df_silver_meta.groupBy(*chave).count()
    .filter(F.col("count") > 1).count()
)
print(f"Chaves duplicadas antes da deduplicação: {dups_antes}")
print(f"Chaves duplicadas após deduplicação: {dups_depois}")

# 2) Domínios das dimensões
uf_invalida = df_silver_meta.filter(
    F.col("sigla_uf").isNotNull() & ~F.col("sigla_uf").isin(*ufs_validas)
).count()
print(f"Siglas de UF inválidas: {uf_invalida}")

print("Redes não reconhecidas:")
(
    df_silver_meta.filter(F.col("rede").isNotNull() & ~F.col("rede").isin(*redes_conhecidas))
    .groupBy("rede").count().orderBy(F.desc("count"))
    .show(100, truncate=False)
)

# 3) Nulos nas colunas
for coluna in colunas_esperadas:
    quantidade = df_silver_meta.filter(F.col(coluna).isNull()).count()
    print(f"Nulos em '{coluna}': {quantidade}")

# 4) Percentuais fora da faixa esperada [0, 100]
for coluna in colunas_percentuais:
    quantidade = df_silver_meta.filter(
        F.col(coluna).isNotNull()
        & ((F.col(coluna) < 0) | (F.col(coluna) > 100) | F.isnan(coluna))
    ).count()
    print(f"Valores preenchidos fora de [0, 100] ou NaN em '{coluna}': {quantidade}")

# 5) As metas anuais devem permanecer iguais ou crescer ao longo do horizonte
meta_nao_monotona = df_silver_meta.filter(
    (F.col("meta_alfabetizacao_2025") < F.col("meta_alfabetizacao_2024"))
    | (F.col("meta_alfabetizacao_2026") < F.col("meta_alfabetizacao_2025"))
    | (F.col("meta_alfabetizacao_2027") < F.col("meta_alfabetizacao_2026"))
    | (F.col("meta_alfabetizacao_2028") < F.col("meta_alfabetizacao_2027"))
    | (F.col("meta_alfabetizacao_2029") < F.col("meta_alfabetizacao_2028"))
    | (F.col("meta_alfabetizacao_2030") < F.col("meta_alfabetizacao_2029"))
).count()
print(f"Linhas com metas decrescentes: {meta_nao_monotona}")

# 6) Taxa e participação devem estar preenchidas ou nulas em conjunto
nulidade_resultado_inconsistente = df_silver_meta.filter(
    F.col("taxa_alfabetizacao").isNull() != F.col("percentual_participacao").isNull()
).count()
print(
    "Linhas com nulidade inconsistente entre taxa e participação: "
    f"{nulidade_resultado_inconsistente}"
)

# 7) Completude das metas
qtd_metas_preenchidas = sum(
    F.when(F.col(coluna).isNotNull(), F.lit(1)).otherwise(F.lit(0))
    for coluna in colunas_meta
)
print("Quantidade de metas preenchidas por ano:")
(
    df_silver_meta.withColumn("qtd_metas_preenchidas", qtd_metas_preenchidas)
    .groupBy("ano", "qtd_metas_preenchidas").count()
    .orderBy("ano", "qtd_metas_preenchidas")
    .show(100, truncate=False)
)

# 8) Cobertura das 27 UFs por ano
print("Quantidade de UFs distintas por ano:")
(
    df_silver_meta.groupBy("ano")
    .agg(F.countDistinct("sigla_uf").alias("qtd_ufs"))
    .orderBy("ano").show()
)

print(f"Linhas Bronze: {qtd_bronze} -> Linhas Silver: {qtd_silver}")

=== Relatório de Qualidade — silver.meta_alfabetizacao_uf ===


Chaves duplicadas antes da deduplicação: 0
Chaves duplicadas após deduplicação: 0
Siglas de UF inválidas: 0
Redes não reconhecidas:
+----+-----+
|rede|count|
+----+-----+
+----+-----+

Nulos em 'ano': 0
Nulos em 'sigla_uf': 0
Nulos em 'rede': 0
Nulos em 'taxa_alfabetizacao': 4
Nulos em 'meta_alfabetizacao_2024': 9
Nulos em 'meta_alfabetizacao_2025': 3
Nulos em 'meta_alfabetizacao_2026': 2
Nulos em 'meta_alfabetizacao_2027': 2
Nulos em 'meta_alfabetizacao_2028': 2
Nulos em 'meta_alfabetizacao_2029': 2
Nulos em 'meta_alfabetizacao_2030': 2
Nulos em 'percentual_participacao': 4
Nulos em '_ingestao_timestamp': 0
Nulos em '_fonte': 0
Valores preenchidos fora de [0, 100] ou NaN em 'taxa_alfabetizacao': 0
Valores preenchidos fora de [0, 100] ou NaN em 'meta_alfabetizacao_2024': 0
Valores preenchidos fora de [0, 100] ou NaN em 'meta_alfabetizacao_2025': 0
Valores preenchidos fora de [0, 100] ou NaN em 'meta_alfabetizacao_2026': 0
Valores preenchidos fora de [0, 100] ou NaN em 'meta_alfabetizac

## 6. Armazenamento no BigQuery

In [16]:
(
    df_silver_meta.write.format("bigquery")
    .option("table", par_source_silver_meta)
    .option("writeMethod", "direct")
    .option("clusteredFields", "ano,sigla_uf,rede")
    .mode("overwrite")
    .save()
)

26/08/24 02:31:10 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
                                                                                